# Grover core — the shared building blocks

This notebook contains **all the functions that the other notebooks use** to build and analyse Grover circuits.
Other notebooks load it with

```python
%run grover_core.ipynb
```

which runs every cell here and makes the functions available, so the circuit is defined in exactly one place.
This notebook therefore contains only definitions (nothing is simulated here).

## The big picture

Grover's algorithm searches an unsorted list of $N = 2^n$ items for the ones that are "marked".
Classically you need on average about $N/2$ checks. Grover needs only about $\frac{\pi}{4}\sqrt{N}$ checks.

The algorithm has three parts:

1. **Start in an equal superposition**: apply a Hadamard gate (H) to every qubit, so all $N$ states have the same probability $1/N$.
2. **Repeat $r$ Grover rounds**, each consisting of
   - the **oracle**, which marks the solutions by flipping the sign of their amplitude, and
   - the **diffuser**, which turns that sign flip into a bigger amplitude (probability) for the solutions.
3. **Measure**. With the right number of rounds, the result is very likely to be a solution.

## Notation used in every notebook

| Symbol | Meaning |
|---|---|
| `n` | number of qubits |
| `N = 2**n` | number of possible states (size of the search space) |
| `marked` | the solution(s), written as bitstrings such as `'101'`; one string or a list |
| `k` | number of marked states (solutions) |
| `r` | number of Grover rounds (oracle + diffuser) |

**Bit order.** Qiskit prints measured bitstrings *right to left*: the rightmost character is qubit 0.
The oracle below follows that convention, so the bitstring you pass in is exactly the key you see in the measurement counts.

In [7]:
import math

from qiskit import QuantumCircuit

## 1. Input checks

Small helper functions that stop the program early with a clear message when the input makes no sense:

- `_check_n`: Grover needs at least 2 qubits. With 1 qubit there are only 2 states and the multi-controlled gates used below don't exist.
- `_as_list`: lets every function accept either one bitstring (`'101'`) or a list (`['101', '011']`). Duplicates are removed.
- `_check_marked`: every marked state must have exactly `n` characters, all `0` or `1`.

The leading underscore `_` is a Python convention meaning "internal helper, not meant to be used directly".

In [8]:
def _check_n(n):
    if not isinstance(n, int) or n < 2:
        raise ValueError(
            f"n must be an integer >= 2 (got n={n!r}). "
            "With fewer than 2 qubits there is nothing to search."
        )


def _as_list(marked):
    """Turn one bitstring or a list of bitstrings into a list (no duplicates, order kept)."""
    if isinstance(marked, str):
        marked = [marked]
    return list(dict.fromkeys(marked))


def _check_marked(n, marked):
    """Validate marked state(s) and return them as a list."""
    states = _as_list(marked)
    if not states:
        raise ValueError("At least one marked state is required.")
    for s in states:
        if len(s) != n or set(s) - {"0", "1"}:
            raise ValueError(
                f"Marked state {s!r} is not a {n}-character bitstring of 0s and 1s."
            )
    return states

## 2. The oracle — "mark the solutions"

The oracle is the part that "knows" the answer. It does not tell us the answer directly. Instead it **flips the sign (phase)** of every marked state:

$$|x\rangle \;\to\; -|x\rangle \quad \text{if } x \text{ is marked}, \qquad |x\rangle \;\to\; |x\rangle \quad \text{otherwise.}$$

A sign flip doesn't change any probability by itself (since $(-a)^2 = a^2$), but the diffuser in the next step uses it.

**How one marked state is flipped**, for example `'101'`:

1. Apply an **X gate** (a NOT) to every qubit that should be `0`. Now the marked state looks like `|111⟩`, and every other state looks like something else.
2. Flip the sign of `|11…1⟩` only. That's a **multi-controlled Z gate**: "apply Z to the last qubit if all other qubits are 1". Qiskit doesn't have it directly here, so we build it as **H – MCX – H** (a Hadamard on each side turns a multi-controlled NOT into a multi-controlled Z).
3. Undo the X gates from step 1, so all states go back to their original labels.

`reversed(state)` handles Qiskit's right-to-left bit order: character 0 of the reversed string is qubit 0.

**Several marked states:** we simply repeat the three steps once per marked state. Each repetition flips the sign of exactly one state, so together they flip all of them.

In [9]:
def oracle(n, marked):
    """Phase oracle: flips the sign of every marked basis state (one bitstring or a list)."""
    _check_n(n)
    states = _check_marked(n, marked)

    qc = QuantumCircuit(n, name="Oracle")
    for state in states:
        zeros = [i for i, b in enumerate(reversed(state)) if b == "0"]
        if zeros:
            qc.x(zeros)              # step 1: marked state -> |11...1>
        qc.h(n - 1)                  # step 2: multi-controlled Z = H - MCX - H
        qc.mcx(list(range(n - 1)), n - 1)
        qc.h(n - 1)
        if zeros:
            qc.x(zeros)              # step 3: undo step 1
    return qc

## 3. The diffuser — "inversion about the mean"

After the oracle, the marked states have a negative amplitude and everything else is positive.
The diffuser **reflects every amplitude around the average amplitude**:

$$a \;\to\; 2\cdot\overline{a} - a$$

Because the marked amplitudes are negative, they're far below the average, so the reflection sends them far *above* it.
The unmarked ones are just above the average and end up a little below it. Result: the marked states gain probability and the others lose some.

In circuit form it's built just like the oracle, but for the state `|00…0⟩` and wrapped in Hadamards:

1. H on all qubits (go back from the superposition to `|00…0⟩`),
2. X on all qubits (`|00…0⟩` → `|11…1⟩`),
3. multi-controlled Z (H – MCX – H) to flip the sign of `|11…1⟩`,
4. undo the X gates and the H gates.

(Mathematically this is the operator $2|s\rangle\langle s| - I$, where $|s\rangle$ is the equal superposition, up to an overall sign that doesn't affect any measurement.)

In [10]:
def diffuser(n):
    """Diffusion operator ("inversion about the mean")."""
    _check_n(n)
    qc = QuantumCircuit(n, name="Diffuser")
    qc.h(range(n))
    qc.x(range(n))
    qc.h(n - 1)
    qc.mcx(list(range(n - 1)), n - 1)
    qc.h(n - 1)
    qc.x(range(n))
    qc.h(range(n))
    return qc

## 4. Theory — how many rounds?

Grover's algorithm has a neat geometric picture. With $k$ solutions out of $N$, define the angle

$$\theta = \arcsin\sqrt{k/N}.$$

The starting superposition is tilted by $\theta$ away from "no solution". **Each round rotates the state by a further $2\theta$ towards the solutions.**
After $r$ rounds the angle is $(2r+1)\theta$, so the probability of measuring a solution is

$$P(r) = \sin^2\big((2r+1)\,\theta\big).$$

This is largest when $(2r+1)\theta \approx \pi/2$ (pointing straight at the solutions). Solving for $r$ and rounding to a whole number gives

$$r_{\text{opt}} = \operatorname{round}\!\left(\frac{\pi}{4\theta} - \frac{1}{2}\right).$$

**Why the exact formula?** The well-known rule of thumb $\frac{\pi}{4}\sqrt{N}$ is only an approximation for large $N$.
For $n = 2$ ($N = 4$) it gives $\operatorname{round}(1.57) = 2$ rounds, but the correct answer is **1** round (which gives 100%). Two rounds would overshoot and give only 25%.

If you do more rounds than $r_{\text{opt}}$, you rotate *past* the solutions and the probability drops again. That's why $P(r)$ oscillates like a wave.

- `optimal_iterations(n, k)` returns $r_{\text{opt}}$.
- `theoretical_success(n, r, k)` returns $P(r)$, the ideal success probability we compare our simulations against.

In [11]:
def _theta(n, k):
    _check_n(n)
    N = 2 ** n
    if not 1 <= k <= N:
        raise ValueError(f"k must be between 1 and N={N} (got k={k}).")
    return math.asin(math.sqrt(k / N))


def optimal_iterations(n, k=1):
    """Exact optimal number of Grover rounds for k marked states out of N = 2**n."""
    theta = _theta(n, k)
    return round(math.pi / (4 * theta) - 0.5)


def theoretical_success(n, r, k=1):
    """Ideal probability of measuring a marked state after r rounds: sin^2((2r + 1) * theta)."""
    theta = _theta(n, k)
    return math.sin((2 * r + 1) * theta) ** 2

## 5. The full Grover circuit

Puts the pieces together:

1. H on every qubit (equal superposition),
2. `iterations` rounds of oracle + diffuser,
3. measure every qubit.

If you don't say how many rounds (`iterations=None`), it uses the optimal number for the given number of marked states.
The oracle and diffuser are built once and then reused in every round.

In [12]:
def grover(n, marked, iterations=None):
    """Full Grover circuit with `iterations` rounds (optimal if None), measured at the end."""
    _check_n(n)
    states = _check_marked(n, marked)
    if iterations is None:
        iterations = optimal_iterations(n, len(states))
    if iterations < 0:
        raise ValueError(f"iterations must be >= 0 (got {iterations}).")

    orc = oracle(n, states)
    dif = diffuser(n)

    qc = QuantumCircuit(n)
    qc.h(range(n))
    for _ in range(iterations):
        qc.compose(orc, inplace=True)
        qc.compose(dif, inplace=True)
    qc.measure_all()
    return qc

## 6. Measuring the success rate

A simulator or a real quantum computer runs the circuit many times (**shots**) and returns **counts**: a dictionary such as
`{'101': 966, '000': 9, ...}` saying how often each bitstring was measured.

The **success rate** is the fraction of shots that landed on *any* of the marked states:

$$\text{success rate} = \frac{\text{number of shots that gave a marked state}}{\text{total shots}}.$$

With several marked states we add up the counts of all of them, because finding any one solution counts as a success.

**Example of how the other notebooks use these functions:**

```python
%run grover_core.ipynb
from qiskit import transpile
from qiskit_aer import AerSimulator

sim = AerSimulator()
qc = grover(3, "101")                         # optimal r = 2
counts = sim.run(transpile(qc, sim), shots=1024).result().get_counts()
print(success_rate(counts, "101", 1024))      # about 0.945
print(theoretical_success(3, 2))              # 0.9453...
```

In [13]:
def success_rate(counts, marked, shots):
    """Fraction of shots that landed on ANY of the marked states."""
    hits = sum(counts.get(s, 0) for s in _as_list(marked))
    return hits / shots